# 09 — Memory test: can the reservoir supply the history?

**Why this notebook exists.** In notebooks 07–08 the fly reservoir, all three controls and plain ridge tied. The diagnosis (progress log): the 28 input features already contain lags and rolling means of the target, so the reservoir's fading memory duplicates what the features carry, and the readout learned to switch the states off (RMSE rose steadily with `state_weight`; the best configurations penalised the states almost completely). That experiment cannot show whether the fly's wiring helps or not.

**This notebook gives the reservoir a job only it can do.** The inputs are *history-free*: current-month values only (state-based, non-state and one-sided deaths this month, neighbour conflict this month, number of neighbours, the seven V-Dem indices). All lags, rolling means, maxima, trends, months-since-conflict, streaks and the never-conflict flag are removed. Whatever the model knows about the past must now come from the reservoir's own dynamics.

### Comparisons (all on the same folds, rows and tuning budget)
| Reference | What it is | Role |
|---|---|---|
| **A: ridge, current values only** | no memory at all | the floor the reservoir must beat |
| **B: ridge, full 28 features** | hand-built lags and rolling means (notebook 05) | the ceiling: what explicit history gives |
| **fly / weight-shuffled / rewired / random** | reservoir on current values, readout `[states, current values]` | how much of the gap between A and B each network recovers |

**Memory recovered** = (RMSE A − RMSE network) / (RMSE A − RMSE B). 0 = no memory extracted, 1 = as good as the hand-built history, above 1 = better than it.

### How to read the outcomes
- **All reservoirs ≈ A:** the dynamics are not producing useful memory; the fly adds nothing.
- **All reservoirs recover a lot and tie each other:** the reservoir mechanism works but the fly's wiring is not special (any random network of that size and density does it).
- **Fly recovers more than the controls, by more than the seed spread and with a paired interval excluding 0:** evidence that specific wiring helps. Fly vs shuffled isolates synapse strengths, fly vs rewired isolates specific connections beyond the degree pattern, fly vs random isolates structure of any kind.

### Search space changes (documented because they differ from 07)
`input_scale` 0.05–5 (stronger drive, more nonlinearity), `leak` 0.05–1, `state_weight` 0.05–5 (states may be weighted up to 5× the raw inputs). `rho` stays 0.3–1.4. Same 40 draws for every network. **The locked test set is never scored.** `QUICK_TEST = True` runs 2 configurations at horizon 1.

### Follow-up sections (added 2026-10-02 after the first full run)
Sections 7–10 test *why* the controls tie the fly. **7:** how large a fly advantage the data can still hide (equivalence bounds). **8:** whether the reservoirs' better onset ranking is real (cluster bootstrap on average precision). **9:** where the memory comes from — a linear reservoir (the `input_scale` → 0 limit, since the tanh search preferred the bottom of its range) and two banks of leaky units with **no recurrence at all**. **10:** whether the 96 MBONs beat 96 random units (size-matched readout). Each experiment writes its own log, so the first full run of sections 1–6 is reused and only the new work is computed. Full run of 7–10: roughly 2–3 hours on a laptop.

## Setup

In [ ]:
import sys, json, time, warnings
from pathlib import Path

root = Path.cwd().resolve()
while not (root / ".git").exists():
    if root == root.parent:
        raise FileNotFoundError("Open this notebook from inside the DACSS_ML_FP repo.")
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import average_precision_score

from src.paths import ROOT
from src.features.build import FEATURES
from src.eval.splits import FOLDS
from src.eval.paired import cluster_bootstrap_rmse_diff, cluster_bootstrap_ap_diff
from src.models.common import HORIZONS as ALL_H, KINDS
from src.models.readout import LAMBDAS, ridge_path
from src.models.reservoir_search import (ReservoirLab, sample_configs, tune, best_configs,
                                         SPACE_MEMORY, SPACE_LINEAR, SPACE_BANK, BANKS)
from src.networks.controls import KINDS as NETS

# ------------------------------------------------------------------ settings
QUICK_TEST = False
N_ITER = 40
SEED = 42
N_READOUT = 1024
N_SEEDS = 5
N_BOOT = 2000
N_MECH = 20          # configurations per network in the mechanism section (9)
MECH_SEEDS = 3       # extra seeds for the mechanism section's best configurations
HORIZONS = [1] if QUICK_TEST else ALL_H
if QUICK_TEST:
    N_ITER, N_SEEDS, N_BOOT = 2, 1, 300
    N_MECH, MECH_SEEDS = 2, 1
RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)
sfx = "_quicktest" if QUICK_TEST else ""
LOG = RESULTS / f"memory_log{sfx}.csv"
BASE = RESULTS / f"baseline_predictions{sfx}.parquet"
pd.set_option("display.width", 160, "display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8a8a85", "axes.labelcolor": "#3d3d3a", "xtick.color": "#5f5e5a",
    "ytick.color": "#5f5e5a", "axes.grid": True, "grid.color": "#e5e4df", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "lines.linewidth": 1.8, "font.size": 10, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left",
})

NET_NAMES = {"fly": "fly", "shuffled": "weight-shuffled", "rewired": "degree-preserving rewired", "er": "random (Erdős–Rényi)"}
COLORS = {"A: ridge, current values": "#8a8a85", "B: ridge, full features": "#2a78d6", "fly": "#1baf7a",
          "weight-shuffled": "#eb6834", "degree-preserving rewired": "#9b59b6", "random (Erdős–Rényi)": "#c9a227"}
print(f"QUICK_TEST={QUICK_TEST}  horizons={HORIZONS}  N_ITER={N_ITER}  seeds={N_SEEDS}")

## 1. The history-free input set
The assertion below is the leakage-and-design guard: no input may carry information from before the origin month.

In [ ]:
MEMORY_FEATURES = ["log_sb", "log_ns", "log_os", "log_nbr", "n_land_nbrs",
                   "v2x_polyarchy", "v2x_libdem", "v2x_civlib", "v2x_clphy", "v2x_rule", "v2x_corr", "v2x_regime"]
REMOVED = [f for f in FEATURES if f not in MEMORY_FEATURES]
HISTORY_TOKENS = ("lag", "mean", "max", "trend", "since", "streak", "never")
assert not any(any(t in f for t in HISTORY_TOKENS) for f in MEMORY_FEATURES), "a history feature slipped in"
assert all(any(t in f for t in HISTORY_TOKENS) for f in REMOVED if f not in ("log_ns", "log_os") and not f.startswith("v2x")
           and f not in ("log_sb", "log_nbr", "n_land_nbrs")), "unexpected removal"
print(f"{len(MEMORY_FEATURES)} current-value inputs kept: {MEMORY_FEATURES}")
print(f"{len(REMOVED)} removed: {REMOVED}")

In [ ]:
lab = ReservoirLab(horizons=HORIZONS, n_random=N_READOUT, seed=SEED, features=MEMORY_FEATURES)
print("Inputs (months x countries x features):", lab.inp.U.shape)
for h in HORIZONS:
    print(f"horizon {h:>2}: {len(lab.y[h]):,} forecast rows")

## 2. Reference A: ridge with current values only
This is the model with no memory. It is computed with the same readout code and the same folds as everything else; reference B is the ridge row of `results/leaderboard.csv`.

In [ ]:
KEEP = ["gwno", "country", "month", "y", "kind", "at_risk", "event"]
lb = pd.read_csv(RESULTS / ("leaderboard_quicktest.csv" if QUICK_TEST else "leaderboard.csv"))
refA, A_pred = {}, []
for h in HORIZONS:
    ti, si = lab.loc[h]; Z = lab.inp.R[ti, si]; trm, vam = lab.masks[h]; y = lab.y[h]
    P = ridge_path(Z, y, trm, vam)
    r = np.mean([np.sqrt(np.mean((y[va][:, None] - p) ** 2, axis=0)) for va, p in zip(vam, P)], axis=0)
    j = int(r.argmin()); refA[h] = r[j]
    d = lab.tables[h]
    for f, va, p in zip(FOLDS, vam, P):
        A_pred.append(d.loc[va, KEEP].assign(horizon=h, model="A: ridge, current values", fold=f.name, pred=p[:, j]))
A_pred = pd.concat(A_pred, ignore_index=True)
refB = lb[lb["model"] == "ridge"].set_index("horizon")["rmse"].to_dict()
ref = pd.DataFrame({"A: ridge, current values": refA, "B: ridge, full features": {h: refB[h] for h in HORIZONS}})
ref["gap A - B"] = ref.iloc[:, 0] - ref.iloc[:, 1]
display(ref.round(4))
print("The gap A - B is the amount of forecasting skill that explicit history provides. A reservoir has something to recover only if this gap is clearly positive.")

## 3. Tune the four networks (same 40 configurations each)

In [ ]:
CONFIGS = sample_configs(N_ITER, SEED, SPACE_MEMORY)
print("Search space: rho 0.3-1.4, input_scale 0.05-5 (log), leak 0.05-1, state_weight 0.05-5 (log)")
for kind in NETS:
    tune(lab, kind, CONFIGS, LOG)
log = pd.read_csv(LOG)
print("networks in log:", sorted(log["network"].unique()))

## 4. Leaderboard and memory recovered

In [ ]:
best = best_configs(log)
tab = pd.DataFrame({"A: ridge, current values": pd.Series(refA), "B: ridge, full features": pd.Series(refB)}).loc[HORIZONS]
for kind in NETS:
    tab[NET_NAMES[kind]] = best[(best["network"] == kind) & (best["variant"] == "random")].set_index("horizon")["rmse"]
display(tab.round(4))
rec = tab[[NET_NAMES[k] for k in NETS]].rsub(tab["A: ridge, current values"], axis=0).div(tab["A: ridge, current values"] - tab["B: ridge, full features"], axis=0)
print("Memory recovered = (A - network) / (A - B):  0 = none, 1 = as good as hand-built history")
display(rec.round(3))
print("MBON-only readout (96 units) vs 1,024 random units, fly:")
mb = best[best["network"] == "fly"].pivot(index="horizon", columns="variant", values="rmse")[["random", "mbon"]]
display(mb.rename(columns={"random": "1,024 random units", "mbon": "96 MBONs"}).round(4))
bp = best[(best["variant"] == "random")].copy()
pp = bp["params"].apply(json.loads).apply(pd.Series)
bp = pd.concat([bp[["network", "horizon", "rmse"]], pp, bp[["lambda_id"]].rename(columns={"lambda_id": "ridge lambda id"})], axis=1)
print("Best configurations (check for values at the edge of their range):")
display(bp.sort_values(["horizon", "network"]).round(3))

In [ ]:
g = (log[(log["variant"] == "random")].groupby(["network", "horizon", "config_id", "lambda_id"])["rmse"].mean()
     .groupby(["network", "horizon", "config_id"]).min().reset_index().merge(pd.DataFrame(CONFIGS).assign(config_id=lambda d: d.index), on="config_id"))
rows = []
for (net, h), s in g.groupby(["network", "horizon"]):
    c = s[["rmse", "rho", "input_scale", "leak", "state_weight"]].corr(method="spearman")["rmse"]
    rows.append({"network": NET_NAMES[net], "horizon": h, **{f"rank corr with {k}": c[k] for k in ["rho", "input_scale", "leak", "state_weight"]}})
sens = pd.DataFrame(rows); print("How RMSE depends on each hyperparameter (Spearman correlation across the 40 configurations):")
display(sens[sens["network"] == "fly"].set_index("horizon").drop(columns="network").round(2))

## 5. Seeds and paired bootstrap
The best configuration of each network is re-run with new random input matrices (and, for controls, new random graphs). The paired comparison uses the seed-0 predictions of each network's best configuration, with the same country-cluster bootstrap as notebooks 06 and 08. **RMSE(A) − RMSE(B); negative = A better.**

In [ ]:
SEEDS = RESULTS / f"memory_seeds{sfx}.csv"
if SEEDS.exists():
    seeds = pd.read_csv(SEEDS)
else:
    rows = []
    for kind in NETS:
        for h in HORIZONS:
            b = best[(best["network"] == kind) & (best["variant"] == "random") & (best["horizon"] == h)].iloc[0]
            cfg = json.loads(b["params"]); lam = int(b["lambda_id"])
            for s in range(0, N_SEEDS + 1):
                res = lab.evaluate(kind, cfg, w_seed=s, graph_seed=s, variants=("random",), horizons=[h])
                rows.append({"network": NET_NAMES[kind], "horizon": h, "seed": s, "rmse": res[("random", h)][0][:, lam].mean()})
            print(f"{NET_NAMES[kind]:<28} horizon {h:>2} done", flush=True)
    seeds = pd.DataFrame(rows); seeds.to_csv(SEEDS, index=False)
extra = seeds[seeds["seed"] > 0]
sm = extra.groupby(["horizon", "network"])["rmse"].agg(["mean", "std"]).unstack("network")
print("mean over new seeds:"); display(sm["mean"].round(4)); print("standard deviation over new seeds:"); display(sm["std"].round(4))

In [ ]:
RP = []
for kind in NETS:
    for h in HORIZONS:
        b = best[(best["network"] == kind) & (best["variant"] == "random") & (best["horizon"] == h)].iloc[0]
        cfg = json.loads(b["params"]); lam = int(b["lambda_id"])
        P = lab.evaluate(kind, cfg, w_seed=0, graph_seed=0, variants=("random",), horizons=[h], predictions=True)[("random", h)][2]
        d = lab.tables[h]
        for f, va, p in zip(FOLDS, lab.masks[h][1], P):
            RP.append(d.loc[va, KEEP].assign(horizon=h, model=NET_NAMES[kind], fold=f.name, pred=p[:, lam]))
RP = pd.concat(RP, ignore_index=True)
base = pd.read_parquet(BASE); B_pred = base[base["model"] == "ridge"].assign(model="B: ridge, full features")
ALLP = pd.concat([A_pred, B_pred, RP], ignore_index=True)
RP.to_parquet(RESULTS / f"memory_predictions{sfx}.parquet")

A_, B_ = "A: ridge, current values", "B: ridge, full features"
pairs = [("fly", A_), ("fly", B_), (B_, A_), ("fly", "weight-shuffled"), ("fly", "degree-preserving rewired"), ("fly", "random (Erdős–Rényi)")]
rows = []
for h in HORIZONS:
    wide = ALLP[ALLP["horizon"] == h].pivot_table(index=["gwno", "month"], columns="model", values="pred")
    yy = ALLP[(ALLP["horizon"] == h) & (ALLP["model"] == A_)].set_index(["gwno", "month"])["y"]
    wide = wide.join(yy).reset_index().dropna()
    for a, b_ in pairs:
        r = cluster_bootstrap_rmse_diff(wide["y"], wide[a], wide[b_], wide["gwno"], B=N_BOOT, seed=SEED)
        rows.append({"horizon": h, "A": a, "B": b_, **r})
paired = pd.DataFrame(rows); paired["significant"] = (paired["lo"] > 0) | (paired["hi"] < 0)
display(paired.round(4))
paired.round(5).to_csv(RESULTS / f"paired_memory{sfx}.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
names = [NET_NAMES[k] for k in NETS]
ax = axes[0]
for i, n in enumerate(names):
    s = extra[extra["network"] == n].groupby("horizon")["rmse"].agg(["mean", "std"]).reindex(HORIZONS)
    ax.errorbar(np.array(HORIZONS) + (i - 1.5) * 0.25, s["mean"], yerr=s["std"].fillna(0), fmt="o", capsize=3, color=COLORS[n], label=n)
for m in [A_, B_]:
    ax.plot(HORIZONS, tab[m], color=COLORS[m], ls="--", lw=1.4, label=m)
ax.set_xticks(HORIZONS); ax.set_xlabel("horizon (months)"); ax.set_ylabel("validation RMSE"); ax.set_title("History-free inputs: RMSE by horizon")
ax.legend(frameon=False, fontsize=7, ncol=2)
ax = axes[1]
for i, n in enumerate(names):
    ax.plot(HORIZONS, rec[n], marker="o", color=COLORS[n], label=n)
ax.axhline(0, color="#8a8a85", lw=1); ax.axhline(1, color="#2a78d6", lw=1, ls="--")
ax.set_xticks(HORIZONS); ax.set_xlabel("horizon (months)"); ax.set_ylabel("memory recovered"); ax.set_title("Share of the A-to-B gap closed")
plt.tight_layout(); plt.show()

## 6. Where is the gain? Row type and onset detection

In [ ]:
show = [A_, B_] + names
rows = []
for (h, m), p in ALLP[ALLP["model"].isin(show)].groupby(["horizon", "model"]):
    r = {"horizon": h, "model": m, "all rows": np.sqrt(np.mean((p["y"] - p["pred"]) ** 2))}
    for k in KINDS:
        s = p[p["kind"] == k]; r[k] = np.sqrt(np.mean((s["y"] - s["pred"]) ** 2))
    a = p[p["at_risk"]]; r["onset AP"] = average_precision_score(a["event"], a["pred"])
    rows.append(r)
bykind = pd.DataFrame(rows)
for h in HORIZONS:
    print(f"Horizon {h}: RMSE by row type and onset average precision (pooled folds 1-5)")
    display(bykind[bykind["horizon"] == h].drop(columns="horizon").set_index("model").reindex(show).round(3))
bykind.round(4).to_csv(RESULTS / f"memory_by_kind{sfx}.csv", index=False)

## 7. How tight is the null? Equivalence bounds
"Intervals span 0" invites the reply that the test had too little power. The intervals are narrow enough to say more: the **lower** end of each fly-vs-control interval is the largest fly advantage still consistent with the data. Dividing it by the A − B gap (pooled RMSE, the same rows) expresses it as a share of the skill that explicit history provides.

In [ ]:
names = [NET_NAMES[k] for k in NETS]
gapAB = paired[(paired["A"] == B_) & (paired["B"] == A_)].set_index("horizon")["diff"].abs()
rows = []
for h in HORIZONS:
    for ctrl in names[1:]:
        r = paired[(paired["horizon"] == h) & (paired["A"] == "fly") & (paired["B"] == ctrl)].iloc[0]
        adv = max(0.0, -r["lo"])
        rows.append({"horizon": h, "control": ctrl, "fly - control": r["diff"], "lo": r["lo"], "hi": r["hi"],
                     "largest plausible fly advantage": adv, "as share of A-B gap": adv / gapAB[h]})
bounds = pd.DataFrame(rows)
display(bounds.round(4))
print(f"Largest fly advantage consistent with the data, any horizon or control: "
      f"{bounds['as share of A-B gap'].max():.1%} of the A-B gap (median {bounds['as share of A-B gap'].median():.1%}).")
bounds.round(5).to_csv(RESULTS / f"memory_bounds{sfx}.csv", index=False)

## 8. Onset detection: is the reservoirs' higher average precision real?
Section 6 showed every reservoir ranking onsets better than reference B at horizons 6 and 12 (and A beating B at 12). Average precision is computed on at-risk rows, using each model's regression prediction as the risk score. The same country-cluster bootstrap is used. **Here a POSITIVE difference means the first model is better** (higher AP), the opposite sign convention to RMSE. Per-fold AP shows whether one fold drives the result.

In [ ]:
AP_PAIRS = [("fly", B_), ("random (Erdős–Rényi)", B_), (A_, B_), ("fly", A_)]
rows, frows = [], []
for h in HORIZONS:
    wide = ALLP[ALLP["horizon"] == h].pivot_table(index=["gwno", "month"], columns="model", values="pred")
    meta = (ALLP[(ALLP["horizon"] == h) & (ALLP["model"] == A_)]
            .set_index(["gwno", "month"])[["at_risk", "event", "fold"]])
    wide = wide.join(meta).reset_index().dropna()
    ar = wide[wide["at_risk"].astype(bool)]
    for a, b_ in AP_PAIRS:
        r = cluster_bootstrap_ap_diff(ar["event"].astype(bool), ar[a], ar[b_], ar["gwno"], B=N_BOOT, seed=SEED)
        rows.append({"horizon": h, "A": a, "B": b_, "events": int(ar["event"].sum()), **r})
    for fold, s in ar.groupby("fold"):
        if s["event"].sum() == 0:
            continue
        frows.append({"horizon": h, "fold": fold, "events": int(s["event"].sum()),
                      **{m: average_precision_score(s["event"].astype(bool), s[m]) for m in [A_, B_, "fly", "random (Erdős–Rényi)"]}})
ap_paired = pd.DataFrame(rows); ap_paired["significant"] = (ap_paired["lo"] > 0) | (ap_paired["hi"] < 0)
print("AP(A) - AP(B), country-cluster bootstrap. Positive = A ranks onsets better.")
display(ap_paired.round(4))
ap_paired.round(5).to_csv(RESULTS / f"memory_onset_ap{sfx}.csv", index=False)
print("Average precision by fold (at-risk rows):")
display(pd.DataFrame(frows).set_index(["horizon", "fold"]).round(3))

## 9. Mechanism: where does the reservoir's memory come from?
Two findings from sections 4–5 motivate this section: the best `input_scale` sat at the **bottom** of its range for every network (RMSE rose with input scale, Spearman about +0.9), and the controls tie the fly. Both suggest the reservoir is working as a *near-linear filter bank*. A linear reservoir's memory is set entirely by the eigenvalues of (1 − leak)·I + leak·Wᵀ, and rescaling every network to the same spectral radius makes those spectra similar, so a tie would be expected.

| Model | What it removes | If it matches the tanh networks... |
|---|---|---|
| **Linear reservoir** (fly and the three controls) | the nonlinearity (tanh → identity; the `input_scale` → 0 limit) | the nonlinear dynamics add nothing |
| **bank_single** | all recurrence: 1,024 independent leaky units, one shared leak rate | one time scale of fading memory is enough |
| **bank_multi** | all recurrence, but each unit has its own leak rate (log-uniform between the tuned slowest rate and 1) | the memory is the leak, not the network |

Same readout (1,024 units plus the 12 current inputs), ridge path, folds and rows as sections 3–5. The budget is `N_MECH` configurations per model, fewer than the 40 used for the tanh networks, which **favours the tanh networks** if anything. The linear search keeps rho below 0.95 so the system cannot diverge. Each experiment has its own log file, resumable like the others.

In [ ]:
LIN_LOG = RESULTS / f"memory_linear_log{sfx}.csv"
BANK_LOG = RESULTS / f"memory_bank_log{sfx}.csv"
LIN_CONFIGS = sample_configs(N_MECH, SEED, SPACE_LINEAR, fixed={"activation": "linear"})
BANK_CONFIGS = sample_configs(N_MECH, SEED, SPACE_BANK)
for kind in NETS:
    tune(lab, kind, LIN_CONFIGS, LIN_LOG, variants=("random",))
for kind in BANKS:
    tune(lab, kind, BANK_CONFIGS, BANK_LOG, variants=("random",))
best_lin = best_configs(pd.read_csv(LIN_LOG)); best_bank = best_configs(pd.read_csv(BANK_LOG))

In [ ]:
mech = tab[[A_, B_]].copy()
for kind in NETS:
    mech[f"{NET_NAMES[kind]} (tanh)"] = tab[NET_NAMES[kind]]
for kind in NETS:
    mech[f"{NET_NAMES[kind]} (linear)"] = best_lin[best_lin["network"] == kind].set_index("horizon")["rmse"]
for kind in BANKS:
    mech[kind] = best_bank[best_bank["network"] == kind].set_index("horizon")["rmse"]
print("Validation RMSE (mean over folds 1-5):")
display(mech.T.round(4))
rec_m = mech.drop(columns=[A_, B_]).rsub(mech[A_], axis=0).div(mech[A_] - mech[B_], axis=0)
print("Memory recovered = (A - model) / (A - B):")
display(rec_m.T.round(3))
print("Best configurations (look for values at the edge of their range):")
for name, b in [("linear", best_lin), ("banks", best_bank)]:
    pp = b["params"].apply(json.loads).apply(pd.Series)
    display(pd.concat([b[["network", "horizon", "rmse", "lambda_id"]], pp.drop(columns=["activation"], errors="ignore")], axis=1)
            .sort_values(["horizon", "network"]).round(3))

### 9b. Seeds and paired tests for the mechanism models
Best configurations re-run with `MECH_SEEDS` new input matrices (and, for the banks, new leak draws; for ER, new graphs). The paired tests answer three questions: does removing the nonlinearity hurt (fly tanh vs fly linear)? Does the fly's wiring help once the reservoir is linear (fly linear vs ER linear)? Does any network beat no recurrence at all (fly tanh vs bank_multi), and how much history does the bank recover on its own (bank_multi vs A and vs B)?

In [ ]:
MSEEDS = RESULTS / f"memory_mech_seeds{sfx}.csv"
MECH_RUNS = [("fly", "linear", best_lin), ("er", "linear", best_lin), ("bank_single", "bank", best_bank), ("bank_multi", "bank", best_bank)]
def mech_label(kind, family):
    return f"{NET_NAMES[kind]} (linear)" if family == "linear" else kind
def best_row(b, kind, h):
    r = b[(b["network"] == kind) & (b["variant"] == "random") & (b["horizon"] == h)].iloc[0]
    return json.loads(r["params"]), int(r["lambda_id"])

if MSEEDS.exists():
    mseeds = pd.read_csv(MSEEDS)
else:
    rows = []
    for kind, fam, b in MECH_RUNS:
        for h in HORIZONS:
            cfg, lam = best_row(b, kind, h)
            for s in range(1, MECH_SEEDS + 1):
                res = lab.evaluate(kind, cfg, w_seed=s, graph_seed=s, variants=("random",), horizons=[h])
                rows.append({"model": mech_label(kind, fam), "horizon": h, "seed": s, "rmse": res[("random", h)][0][:, lam].mean()})
        print(f"{mech_label(kind, fam):<24} seeds done", flush=True)
    mseeds = pd.DataFrame(rows); mseeds.to_csv(MSEEDS, index=False)
print("Mean (sd) RMSE over new seeds:")
display(mseeds.groupby(["horizon", "model"])["rmse"].agg(["mean", "std"]).unstack("model").round(4))

MP = []
for kind, fam, b in MECH_RUNS:
    for h in HORIZONS:
        cfg, lam = best_row(b, kind, h)
        P = lab.evaluate(kind, cfg, variants=("random",), horizons=[h], predictions=True)[("random", h)][2]
        d = lab.tables[h]
        for f, va, p in zip(FOLDS, lab.masks[h][1], P):
            MP.append(d.loc[va, KEEP].assign(horizon=h, model=mech_label(kind, fam), fold=f.name, pred=p[:, lam]))
MP = pd.concat(MP, ignore_index=True)
MP.to_parquet(RESULTS / f"memory_mech_predictions{sfx}.parquet")
ALLM = pd.concat([ALLP, MP], ignore_index=True)

FL, EL = "fly (linear)", "random (Erdős–Rényi) (linear)"
mpairs = [("fly", FL), (FL, EL), ("fly", "bank_multi"), ("bank_multi", "bank_single"), ("bank_multi", A_), ("bank_multi", B_)]
rows = []
for h in HORIZONS:
    wide = ALLM[ALLM["horizon"] == h].pivot_table(index=["gwno", "month"], columns="model", values="pred")
    yy = ALLM[(ALLM["horizon"] == h) & (ALLM["model"] == A_)].set_index(["gwno", "month"])["y"]
    wide = wide.join(yy).reset_index().dropna()
    for a, b_ in mpairs:
        r = cluster_bootstrap_rmse_diff(wide["y"], wide[a], wide[b_], wide["gwno"], B=N_BOOT, seed=SEED)
        rows.append({"horizon": h, "A": a, "B": b_, **r})
mpaired = pd.DataFrame(rows); mpaired["significant"] = (mpaired["lo"] > 0) | (mpaired["hi"] < 0)
print("RMSE(A) - RMSE(B), pooled rows; negative = A better:")
display(mpaired.round(4))
mpaired.round(5).to_csv(RESULTS / f"paired_mechanism{sfx}.csv", index=False)

## 10. Does the biological readout beat a random one of the same size?
The 96 MBONs are the fly's own readout layer. Section 4 compared them with **1,024** random units, which mixes "which units" with "how many". Here five sets of 96 randomly chosen units are tuned on the same 40 configurations as the MBON readout (the fly network, same inputs). If the MBONs beat most random 96-unit sets, the biological readout picks out unusually useful dimensions; if they sit inside the random range, they do not.

In [ ]:
R96_LOG = RESULTS / f"memory_r96_log{sfx}.csv"
R96 = lab.add_random96(5)
tune(lab, "fly", CONFIGS, R96_LOG, variants=tuple(R96))
b96 = best_configs(pd.read_csv(R96_LOG))
cmp96 = b96.pivot(index="horizon", columns="variant", values="rmse")[R96]
out96 = pd.DataFrame({"96 MBONs": mb["mbon"],
                      "random 96: mean": cmp96.mean(axis=1), "random 96: best": cmp96.min(axis=1), "random 96: worst": cmp96.max(axis=1),
                      "1,024 random units": mb["random"]})
out96["MBON rank (1 = best of 6)"] = [int((cmp96.loc[h] < out96.loc[h, "96 MBONs"]).sum()) + 1 for h in out96.index]
display(out96.round(4))
out96.round(5).to_csv(RESULTS / f"memory_mbon_vs_random96{sfx}.csv")

## 11. Reading the results
Fill in after the full run, in this order:

1. **Is there memory to recover?** The A − B gap (section 2). If it is small at some horizon, nothing can be recovered there.
2. **Do the reservoirs recover it?** Memory-recovered table (section 4). Compare with the paired interval for fly vs A (should be clearly negative) and fly vs B.
3. **Do the controls tie the fly?** Fly vs each control: a gap bigger than the seed spread *and* an interval excluding 0 is required before saying the wiring matters. Report the bound from section 7 as "a fly advantage larger than X% of the history gap is ruled out".
4. **Sensitivity table:** does RMSE now depend on `state_weight`, `leak` and `rho`? In 07 it depended only on `state_weight`, because the states were being ignored.
5. **Edge of range:** any best value at a boundary means the range should be widened. `input_scale` was at the bottom edge in the first run; section 9's linear reservoir is the limit of that direction.
6. **Onset AP (section 8):** only call the reservoirs' onset gain real if the bootstrap interval excludes 0 *and* it holds in most folds.
7. **Mechanism (section 9):** linear ≈ tanh → the nonlinearity adds nothing. bank_multi ≈ networks → the memory comes from leaky integration over many time scales, not from recurrence or wiring. Fly linear vs ER linear repeats the wiring test without the nonlinearity.
8. **MBONs (section 10):** a rank of 1 at most horizons is needed before saying evolution picked a good readout.
9. **Report both experiments** (07–08 full features, 09 history-free). Whatever the outcome, a controlled result is reportable.